# gold_vuln_package_build
`silver_advisory_ghsa` + `silver_advisory_ghsa_affected` + `silver_vuln_alias` + `silver_package` → `gold_vuln_package`.
Grain: `ghsa_id` × `package_key` × `range`. **오늘 변경된 GHSA만** 대상으로 하고(`JOB_GHSA_DAILY` 완료 후 트리거), 해당 GHSA의 기존 행을 지우고 새로 넣는다 — `silver_advisory_ghsa_affected`와 동일한 "부모 키 단위 전체 교체" 패턴(03_Gold_Schema.md §3.7, §3.10).

**Business Key**: `range_idx`가 아니라 `ghsa_id + package_key + normalized_vulnerable_version_range + first_patched_version`을 해시한 `vuln_package_key`.

In [ ]:
%run 00_gold_common

In [ ]:
affected = spark.table(TBL["advisory_ghsa_affected"])

if target_ghsa_ids:
    scoped_affected = affected.filter(F.col("ghsa_id").isin(target_ghsa_ids))
else:
    # 파이프라인이 JOB_GHSA_DAILY와 동일한 batch_id를 전달했다는 전제 하에, 오늘 배치가 건드린 GHSA만 대상으로 삼는다.
    scoped_affected = affected.filter(F.col("batch_id") == batch_id)

scope_ghsa_ids = [r["ghsa_id"] for r in scoped_affected.select("ghsa_id").distinct().collect()]


In [ ]:
normalized_range = F.trim(F.regexp_replace(F.col("vulnerable_version_range"), r"\s+", " "))

scoped_affected = scoped_affected.withColumn("normalized_vulnerable_version_range", normalized_range)

vuln_package_key = content_hash(
    [F.col("ghsa_id"), F.col("package_key"), F.col("normalized_vulnerable_version_range"),
     F.col("first_patched_version")],
    length=32,
)
scoped_affected = scoped_affected.withColumn("vuln_package_key", vuln_package_key)


In [ ]:
advisory_latest = spark.table(TBL["advisory_ghsa"]).filter(F.col("valid_to").isNull()).select(
    "ghsa_id", "summary", "description", "severity", "cvss_score", "cvss_vector", "cwe_ids",
    "published_at", "withdrawn_at",
).withColumn("is_withdrawn", F.col("withdrawn_at").isNotNull())

alias_cve = spark.table(TBL["vuln_alias"]).filter(F.col("id_type") == "CVE")
cve_ids_per_ghsa = alias_cve.groupBy(F.col("canonical_key").alias("ghsa_id")).agg(
    F.sort_array(F.collect_set("id_value")).alias("cve_ids")
)

package_meta = spark.table(TBL["package"]).select("package_key", "ecosystem", "package_name")


In [ ]:
gold_df = (
    scoped_affected
    .drop("ecosystem", "package_name")  # silver_package가 더 정제된 값을 갖고 있으므로 그쪽을 신뢰한다
    .join(package_meta, on="package_key", how="left")
    .join(advisory_latest, on="ghsa_id", how="left")
    .join(cve_ids_per_ghsa, on="ghsa_id", how="left")
    .withColumn("cve_ids", F.coalesce(F.col("cve_ids"), F.array().cast("array<string>")))
    .withColumn("source_batch_ids", F.array(F.col("batch_id")))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
    .select(
        "vuln_package_key", "ghsa_id", "package_key", "ecosystem", "package_name",
        "normalized_vulnerable_version_range", "first_patched_version", "range_idx",
        "cve_ids", "summary", "description", "severity", "cvss_score", "cvss_vector", "cwe_ids",
        "published_at", "withdrawn_at", "is_withdrawn",
        "source_batch_ids", "batch_id", "ingested_at",
    )
)
gold_count = gold_df.count()


In [ ]:
replace_by_key(gold_df, TBL["gold_vuln_package"], "ghsa_id", scope_ghsa_ids)

log_batch_run("gold_vuln_package_build", "SUCCESS", record_count=gold_count)
log_dq_metric("gold_vuln_package_build", "scoped_ghsa_count", len(scope_ghsa_ids), passed=True)
print(f"[gold_vuln_package_build] rows={gold_count} scoped_ghsa={len(scope_ghsa_ids)} batch_id={batch_id}")
